# 07: Hai baseline cơ bản trên cùng cách chia 8/1/1 (ItemKNN, logistic regression)

Mục đích: kiểm tra điểm cao của hybrid tuần 3 (~0.44 NDCG@5) đến từ cách dữ liệu được dựng, không phải do rò rỉ hay lỗi cách chia, bằng hai mô hình có sẵn đơn giản không dùng chung code mô hình nào với hybrid.

| Baseline | Thư viện | Thấy | Không thấy |
|---|---|---|---|
| **ItemKNN** (item-item cosine, Sarwar et al. 2001) | `implicit.nearest_neighbours.CosineRecommender` | 20 booking gần nhất của user trước mục tiêu | destination truy vấn, mọi ngữ cảnh |
| **Multinomial logistic regression** | `sklearn.linear_model.LogisticRegression(solver="saga")` | destination truy vấn + ngữ cảnh chuyến đi của lượt tìm kiếm hiện tại (one-hot) | lịch sử user, mọi trường khách sạn đã đặt (`tgt_*`), `ctx_user_city`, `ctx_user_region`, khoảng cách |

Cách chia: chia theo thời gian toàn cục 80/10/10 trên các mục tiêu next-booking (mọi booking trừ booking đầu tiên của user), cùng cách chia của lượt chạy SMLP4Rec plain (RecBole `RS: [0.8, 0.1, 0.1]`, `order: TO`) (được dựng lại ở đây bởi `load_bookings`, cùng quy tắc). Các booking đầu tiên (L = 0) trong cửa sổ valid / test tạo thành lát cắt cold, như notebook 06. Kích thước các lát cắt được kiểm tra với JSON của notebook 06.

Dữ liệu huấn luyện cho cả hai mô hình: các booking nằm hoàn toàn trước `cut = thời điểm mục tiêu train cuối - 128 s`, cùng điểm cắt với destination prior. Siêu tham số (`K` của ItemKNN, `C` của logistic regression) được chọn trên valid theo trung bình Recall@5 + NDCG@5 từng dòng (mục tiêu của hybrid tuần 3); test được chấm một lần với lựa chọn đó.

Hòa điểm: các cột được sắp theo độ phổ biến thời kỳ train trước khi xếp hạng, nên hòa điểm được phá theo hướng cluster phổ biến hơn (ItemKNN cho 0 với các cluster không liên quan đến lịch sử). Mục 7 báo cáo quy tắc hòa điểm có thể làm dịch các con số bao nhiêu.

Output: `results/week4_rebuild/basic_baselines.json`; thứ hạng từng dòng trong `data/interim/basic_baselines_ranks_<split>.parquet` (gitignored).

In [1]:
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

import implicit
import sklearn
from implicit.nearest_neighbours import CosineRecommender
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from nbp.eval.metrics import summarize, target_rank

BOOKINGS = ROOT / "data" / "interim" / "expedia_bookings.parquet"
REF_JSON = ROOT / "results" / "week3_implementation" / "smlprec_expedia_seen_unseen_users.json"
OUT_JSON = ROOT / "results" / "week4_rebuild" / "basic_baselines.json"
RANKS_DIR = ROOT / "data" / "interim"

N_CLUSTERS = 100
MAX_HIST = 20  # MAX_ITEM_LIST_LENGTH of the plain SMLP4Rec run (configs/smlprec_expedia.yaml)
TIME_MARGIN = 128  # s; prior / training cut sits this far before the last train target (notebooks 03-06)
SPLIT = (0.8, 0.1, 0.1)  # eval_args.split of configs/smlprec_expedia.yaml, ordered by time
SEED = 2022
K_GRID = [5, 10, 20, 50, 100]  # ItemKNN neighbours kept per item (100 = all)
C_GRID = [0.1, 1.0, 10.0]  # inverse L2 strength of the logistic regression
SAGA_EPOCHS = 3  # user decision 2026-10-05: 3 passes over the training data
SAGA_TOL = 0.0  # never stop early: exactly SAGA_EPOCHS epochs
LEAD_BINS = [-1, 0, 2, 6, 13, 29, 59, 89, 179, 364, 10_000]  # days between search and check-in
STAY_CLIP = 14  # nights; longer stays share one bucket
LR_FEATURES = [  # query-time fields only
    "ctx_dest_id", "ctx_dest_type", "ctx_checkin_month", "stay_bucket", "lead_bucket",
    "ctx_adults", "ctx_children", "ctx_rooms", "ctx_is_package", "ctx_channel", "ctx_is_mobile",
    "ctx_site_name", "ctx_posa_continent", "ctx_user_country",
]
FORBIDDEN = ("tgt_", "ctx_user_city", "ctx_user_region", "distance", "user_id", "item_id", "src_row", "aux_cnt")
print("implicit", implicit.__version__, "| scikit-learn", sklearn.__version__)

implicit 0.7.3 | scikit-learn 1.9.1


## 1. Dữ liệu và cách chia

In [2]:
def load_bookings() -> pd.DataFrame:
    # bookings sorted by (time, src_row); pos = per-user index (0 = first booking);
    # split = global temporal 80/10/10 over next-booking targets (pos > 0), first bookings are never targets
    d = pd.read_parquet(BOOKINGS, columns=["user_id", "item_id", "timestamp", "src_row", "ctx_dest_id"])
    d["ts"] = ((d["timestamp"] - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)).astype("int64")
    d = d.sort_values(["ts", "user_id", "src_row"], kind="stable").reset_index(drop=True)
    d["item_id"] = d["item_id"].astype("int64")
    d["pos"] = d.groupby("user_id").cumcount()
    d["split"] = "first_booking"
    tgt = np.flatnonzero(d["pos"].to_numpy() > 0)
    n_tr, n_va = int(SPLIT[0] * len(tgt)), int(SPLIT[1] * len(tgt))
    d.loc[tgt[:n_tr], "split"] = "train"
    d.loc[tgt[n_tr : n_tr + n_va], "split"] = "valid"
    d.loc[tgt[n_tr + n_va :], "split"] = "test"
    return d


t0 = time.time()
b = load_bookings()
ctx = pd.read_parquet(
    BOOKINGS,
    columns=["src_row", "ctx_dest_type", "ctx_checkin_month", "ctx_stay_nights", "ctx_lead_days",
             "ctx_adults", "ctx_children", "ctx_rooms", "ctx_is_package", "ctx_channel", "ctx_is_mobile",
             "ctx_site_name", "ctx_posa_continent", "ctx_user_country"],
)
assert ctx["src_row"].is_unique
b = b.merge(ctx, on="src_row", how="left", validate="one_to_one")
b = b.sort_values(["ts", "user_id", "src_row"], kind="stable").reset_index(drop=True)

rng_t = b[b["split"].isin(["train", "valid", "test"])].groupby("split")["ts"].agg(["min", "max"])
t_train_max, t_valid_max = int(rng_t.loc["train", "max"]), int(rng_t.loc["valid", "max"])
cut = t_train_max - TIME_MARGIN

# cold rows: a user's first booking (pos 0) inside the valid / test window
first = b["pos"] == 0
b.loc[first & (b["ts"] > t_train_max) & (b["ts"] <= t_valid_max), "split"] = "valid_cold"
b.loc[first & (b["ts"] > t_valid_max), "split"] = "test_cold"
seen_users = set(b.loc[b["split"] == "train", "user_id"].unique())
b["seen_user"] = b["user_id"].isin(seen_users)

print(b["split"].value_counts().to_string())
print("time ranges (unix):\n", rng_t.to_string())
print(f"cut = {cut}; bookings before cut (training data of both models): {(b['ts'] < cut).sum():,}")
print(f"{time.time() - t0:.0f}s")

split
train            1710952
first_booking     713754
valid             213869
test              213869
valid_cold         50769
test_cold          49462
time ranges (unix):
               min         max
split                        
test   1416397083  1420070370
train  1357518917  1412777354
valid  1412777363  1416397080
cut = 1412777226; bookings before cut (training data of both models): 2,424,679
26s


In [3]:
# Same events as the reference run (notebook 06)? Compare slice sizes with its JSON.
ref = json.loads(REF_JSON.read_text(encoding="utf-8"))
PLAIN, HYB = "plain SMLP4Rec", "hybrid (SMLP4Rec + prior + sameDest)"
EVAL = {s: {"warm": b[b["split"] == s].reset_index(drop=True), "cold": b[b["split"] == s + "_cold"].reset_index(drop=True)}
        for s in ("valid", "test")}

size_check = {}
for s, d in EVAL.items():
    w, c = d["warm"], d["cold"]
    mine = {
        "seen users": int(w["seen_user"].sum()),
        "unseen, L>=1": int((~w["seen_user"]).sum()),
        "unseen, L=0 (first booking)": int(len(c)),
        "all events": int(len(w) + len(c)),
    }
    theirs = {k: ref["results"][s][k][PLAIN]["n"] for k in mine}
    size_check[s] = {k: {"here": mine[k], "reference": theirs[k], "diff": mine[k] - theirs[k]} for k in mine}
    # INFORMATIONAL: the reference run used the uncollapsed data, so sizes differ by the collapsed bursts.
print(pd.DataFrame({(s, k): v for s, d in size_check.items() for k, v in d.items()}).T.to_string())

                                     here  reference  diff
valid seen users                   157601     161151 -3550
      unseen, L>=1                  56268      57519 -1251
      unseen, L=0 (first booking)   50769      50803   -34
      all events                   264638     269473 -4835
test  seen users                   139815     143025 -3210
      unseen, L>=1                  74054      75645 -1591
      unseen, L=0 (first booking)   49462      49443    19
      all events                   263331     268113 -4782


## 2. Hàm dùng chung: thứ tự phá hòa theo popularity, xếp hạng, ma trận lịch sử

In [4]:
train_rows = b[b["ts"] < cut]
pop = np.bincount(train_rows["item_id"].to_numpy(), minlength=N_CLUSTERS).astype(np.float64)
pop_share = pop / pop.sum()
order = np.argsort(-pop, kind="stable")  # column j of a permuted matrix = cluster order[j]
inv_order = np.empty(N_CLUSTERS, np.int64)
inv_order[order] = np.arange(N_CLUSTERS)


def rank_pop_ties(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    # rank of the true cluster; ties go to the more popular cluster (train-period popularity)
    return target_rank(np.asarray(scores)[:, order], inv_order[np.asarray(y)])


def joint(r: np.ndarray) -> float:
    # mean per-row Recall@5 + NDCG@5 (selection objective on valid)
    h = r <= 5
    return float((h * (1.0 + 1.0 / np.log2(r + 1))).mean())


# every user's bookings in (user, pos) order, for history lookups
bu = b.sort_values(["user_id", "pos"], kind="stable").reset_index(drop=True)
u_codes, u_start = np.unique(bu["user_id"].to_numpy(), return_index=True)
bu_item, bu_ts = bu["item_id"].to_numpy(), bu["ts"].to_numpy()


def history_matrix(rows: pd.DataFrame) -> tuple[sp.csr_matrix, np.ndarray]:
    # (n, 100) counts of the user's last MAX_HIST bookings strictly before each row, and each row's latest history ts
    start = u_start[np.searchsorted(u_codes, rows["user_id"].to_numpy())]
    p = rows["pos"].to_numpy()
    s, e = start + np.maximum(0, p - MAX_HIST), start + p
    ln = e - s
    idx = np.arange(ln.sum()) - np.repeat(np.cumsum(ln) - ln - s, ln)
    r = np.repeat(np.arange(len(rows)), ln)
    h = sp.csr_matrix((np.ones(len(idx), np.float32), (r, bu_item[idx])), shape=(len(rows), N_CLUSTERS))
    last_ts = np.full(len(rows), -1, np.int64)
    np.maximum.at(last_ts, r, bu_ts[idx])
    return h, last_ts


Y = {s: {k: v["item_id"].to_numpy() for k, v in d.items()} for s, d in EVAL.items()}
HIST = {s: history_matrix(d["warm"]) for s, d in EVAL.items()}
for s, (h, last_ts) in HIST.items():
    w = EVAL[s]["warm"]
    n_hist = np.asarray(h.sum(1)).ravel()
    assert (n_hist >= 1).all(), "a warm row without history"
    assert (n_hist == np.minimum(w["pos"].to_numpy(), MAX_HIST)).all(), "history length mismatch"
    assert (last_ts <= w["ts"].to_numpy()).all(), "history booking after the target"
    print(s, "mean history length", round(float(n_hist.mean()), 3),
          "| rows whose history has a booking at the target's second:", int((last_ts == w["ts"].to_numpy()).sum()))

valid mean history length 7.305 | rows whose history has a booking at the target's second: 1
test mean history length 7.166 | rows whose history has a booking at the target's second: 1


## 3. Dòng tham chiếu: global popularity (không học)

In [5]:
ranks = {s: {"warm": {}, "cold": {}} for s in EVAL}
GLOB = "global popularity"
for s in EVAL:
    for k in ("warm", "cold"):
        y = Y[s][k]
        ranks[s][k][GLOB] = rank_pop_ties(np.tile(pop_share, (len(y), 1)), y)
    print(s, "warm", summarize(ranks[s]["warm"][GLOB]))

valid warm {'n': 213869, 'recall@5': 0.1411, 'recall@10': 0.2311, 'recall@20': 0.3964, 'ndcg@5': 0.0916, 'ndcg@10': 0.1204, 'ndcg@20': 0.1618}


test warm {'n': 213869, 'recall@5': 0.1487, 'recall@10': 0.2488, 'recall@20': 0.3968, 'ndcg@5': 0.0944, 'ndcg@10': 0.1263, 'ndcg@20': 0.1633}


## 4. Baseline 1: ItemKNN (chỉ lịch sử)

Fit trên ma trận đếm user x cluster của các booking trước điểm cắt. ItemKNN không có epoch: một lượt tính độ tương đồng cosine ở dạng đóng. Điểm của một dòng là `history_counts @ similarity` (những gì `CosineRecommender.recommend` tính; được kiểm tra với `recommend()` ở mục 7). Dòng cold (không có lịch sử) quay về global popularity, như lượt chạy SMLP4Rec plain.

In [6]:
ui = sp.csr_matrix(
    (np.ones(len(train_rows), np.float32),
     (pd.factorize(train_rows["user_id"])[0], train_rows["item_id"].to_numpy())),
    shape=(train_rows["user_id"].nunique(), N_CLUSTERS),
)
ui.sum_duplicates()
print("fit matrix", ui.shape, "nnz", ui.nnz)

knn_models, knn_valid = {}, {}
for k_nb in K_GRID:
    m = CosineRecommender(K=k_nb)
    m.fit(ui, show_progress=False)
    knn_models[k_nb] = m
    r = rank_pop_ties((HIST["valid"][0] @ m.similarity).toarray(), Y["valid"]["warm"])
    knn_valid[k_nb] = {"objective": round(joint(r), 4), **summarize(r)}
    print(k_nb, knn_valid[k_nb])
K_PICK = max(K_GRID, key=lambda k_nb: (knn_valid[k_nb]["objective"], -k_nb))
print("picked K =", K_PICK)

KNN = f"ItemKNN (cosine, K={K_PICK}, history only)"
for s in EVAL:
    ranks[s]["warm"][KNN] = rank_pop_ties((HIST[s][0] @ knn_models[K_PICK].similarity).toarray(), Y[s]["warm"])
    ranks[s]["cold"][KNN] = ranks[s]["cold"][GLOB]  # no history: popularity fallback

fit matrix (713747, 100) nnz 1963651


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.030801773071289062 seconds
  warnings.warn(


5 {'objective': 0.5095, 'n': 213869, 'recall@5': 0.3009, 'recall@10': 0.4248, 'recall@20': 0.5642, 'ndcg@5': 0.2086, 'ndcg@10': 0.2485, 'ndcg@20': 0.2837}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.043356895446777344 seconds
  warnings.warn(


10 {'objective': 0.5123, 'n': 213869, 'recall@5': 0.3028, 'recall@10': 0.4325, 'recall@20': 0.5896, 'ndcg@5': 0.2095, 'ndcg@10': 0.2513, 'ndcg@20': 0.291}


...

20 {'objective': 0.5144, 'n': 213869, 'recall@5': 0.3041, 'recall@10': 0.4367, 'recall@20': 0.6056, 'ndcg@5': 0.2103, 'ndcg@10': 0.253, 'ndcg@20': 0.2955}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.04015612602233887 seconds
  warnings.warn(


50 {'objective': 0.5151, 'n': 213869, 'recall@5': 0.3047, 'recall@10': 0.439, 'recall@20': 0.6112, 'ndcg@5': 0.2104, 'ndcg@10': 0.2536, 'ndcg@20': 0.297}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.029918909072875977 seconds
  warnings.warn(


100 {'objective': 0.5152, 'n': 213869, 'recall@5': 0.3047, 'recall@10': 0.4384, 'recall@20': 0.6114, 'ndcg@5': 0.2105, 'ndcg@10': 0.2536, 'ndcg@20': 0.2971}
picked K = 100


## 5. Baseline 2: multinomial logistic regression (destination truy vấn + ngữ cảnh chuyến đi, không lịch sử)

Huấn luyện bằng SAGA đúng 3 epoch (không early stop). Mọi feature đều là hạng mục và được mã hóa one-hot; một giá trị bị làm sạch thành null (vd. không có người lớn, check-in trước ngày tìm kiếm) là một hạng mục riêng; các giá trị chưa thấy trước điểm cắt (vd. destination mới) ánh xạ về khối toàn số 0, nên hệ số chặn mang global popularity.

In [7]:
def lr_frame(d: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame(index=d.index)
    f["stay_bucket"] = d["ctx_stay_nights"].clip(0, STAY_CLIP)
    f["lead_bucket"] = pd.cut(d["ctx_lead_days"], LEAD_BINS, labels=False)
    for c in LR_FEATURES:
        if c not in f:
            f[c] = d[c]
    # fields nulled by cleaning (flag_zero_adults, flag_zero_rooms, flag_ci_before_search, ...) -> own category -1
    return f[LR_FEATURES].astype("Float64").fillna(-1).astype("int64").astype(str)


enc = OneHotEncoder(handle_unknown="ignore", dtype=np.float64)
X_tr = enc.fit_transform(lr_frame(train_rows))
y_tr = train_rows["item_id"].to_numpy()
X_ev = {s: {k: enc.transform(lr_frame(v)) for k, v in d.items()} for s, d in EVAL.items()}
print("train X", X_tr.shape, "| classes", len(np.unique(y_tr)))

lr_models, lr_valid = {}, {}
for c_val in C_GRID:
    t1 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        m = LogisticRegression(solver="saga", C=c_val, max_iter=SAGA_EPOCHS, tol=SAGA_TOL, random_state=SEED)
        m.fit(X_tr, y_tr)
    assert (m.classes_ == np.arange(N_CLUSTERS)).all()
    lr_models[c_val] = m
    r = rank_pop_ties(m.predict_proba(X_ev["valid"]["warm"]), Y["valid"]["warm"])
    lr_valid[c_val] = {"objective": round(joint(r), 4), "n_iter": int(np.max(m.n_iter_)),
                       "fit_seconds": round(time.time() - t1), **summarize(r)}
    print(c_val, lr_valid[c_val])
C_PICK = max(C_GRID, key=lambda c_val: (lr_valid[c_val]["objective"], -c_val))
print("picked C =", C_PICK)

LR = f"Logistic regression (C={C_PICK}, destination + context, no history)"
for s in EVAL:
    for k in ("warm", "cold"):
        ranks[s][k][LR] = rank_pop_ties(lr_models[C_PICK].predict_proba(X_ev[s][k]), Y[s][k])

train X (2424679, 34877) | classes 100


0.1 {'objective': 0.856, 'n_iter': 3, 'fit_seconds': 216, 'n': 213869, 'recall@5': 0.5083, 'recall@10': 0.6958, 'recall@20': 0.8558, 'ndcg@5': 0.3478, 'ndcg@10': 0.4085, 'ndcg@20': 0.4492}


1.0 {'objective': 0.8952, 'n_iter': 3, 'fit_seconds': 259, 'n': 213869, 'recall@5': 0.5316, 'recall@10': 0.7217, 'recall@20': 0.8819, 'ndcg@5': 0.3637, 'ndcg@10': 0.4253, 'ndcg@20': 0.466}


10.0 {'objective': 0.898, 'n_iter': 3, 'fit_seconds': 251, 'n': 213869, 'recall@5': 0.5333, 'recall@10': 0.725, 'recall@20': 0.8848, 'ndcg@5': 0.3647, 'ndcg@10': 0.4268, 'ndcg@20': 0.4674}
picked C = 10.0


## 6. Kết quả: cùng các lát cắt như notebook 06, với các dòng SMLP4Rec plain và hybrid của nó đọc từ JSON

In [8]:
MODELS = [GLOB, KNN, LR]
COLS = ["n", "recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
SLICES = ["all events", "warm (all L>=1 targets)", "seen users", "unseen, L>=1", "unseen, L=0 (first booking)"]


def slices_of(s: str) -> dict:
    w = EVAL[s]["warm"]["seen_user"].to_numpy()
    out = {}
    for name in MODELS:
        rw, rc = ranks[s]["warm"][name], ranks[s]["cold"][name]
        out[name] = {
            "all events": summarize(np.concatenate([rw, rc])),
            "warm (all L>=1 targets)": summarize(rw),
            "seen users": summarize(rw[w]),
            "unseen, L>=1": summarize(rw[~w]),
            "unseen, L=0 (first booking)": summarize(rc),
        }
    return out


results = {s: slices_of(s) for s in EVAL}
for s in ("test", "valid"):
    for sl in SLICES:
        rows = {name: results[s][name][sl] for name in MODELS}
        if sl in ref["results"][s]:
            rows |= {f"[ref nb06, uncollapsed data] {k}": v for k, v in ref["results"][s][sl].items()}
        print(f"\n{s.upper()} | {sl}")
        display(pd.DataFrame(rows).T[COLS])


TEST | all events


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,263331.0,0.1455,0.0919,0.2447,0.1236,0.3906,0.1601
"ItemKNN (cosine, K=100, history only)",263331.0,0.2715,0.1851,0.3983,0.2258,0.5637,0.2674
"Logistic regression (C=10.0, destination + context, no history)",263331.0,0.5198,0.3554,0.7089,0.4166,0.8724,0.4581
"[ref nb06, uncollapsed data] plain SMLP4Rec",268113.0,0.2990,0.2139,0.4206,0.2530,0.5815,0.2935
"[ref nb06, uncollapsed data] hybrid (SMLP4Rec + prior + sameDest)",268113.0,0.5925,0.4349,0.7608,0.4894,0.9027,0.5255



TEST | warm (all L>=1 targets)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,213869.0,0.1487,0.0944,0.2488,0.1263,0.3968,0.1633
"ItemKNN (cosine, K=100, history only)",213869.0,0.3038,0.2091,0.4380,0.2522,0.6101,0.2955
"Logistic regression (C=10.0, destination + context, no history)",213869.0,0.5222,0.3568,0.7115,0.4181,0.8739,0.4594



TEST | seen users


...

,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,157601.0,0.1435,0.0933,0.2347,0.1225,0.4052,0.1653
"ItemKNN (cosine, K=100, history only)",157601.0,0.3134,0.2132,0.4530,0.2582,0.6281,0.3023
"Logistic regression (C=10.0, destination + context, no history)",157601.0,0.5393,0.3699,0.7301,0.4318,0.8876,0.4718
"[ref nb06, uncollapsed data] plain SMLP4Rec",161151.0,0.3433,0.2455,0.4740,0.2875,0.6440,0.3304
"[ref nb06, uncollapsed data] hybrid (SMLP4Rec + prior + sameDest)",161151.0,0.6277,0.4671,0.7912,0.5201,0.9206,0.5531



VALID | unseen, L>=1


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,56268.0,0.1345,0.0867,0.2212,0.1145,0.3720,0.1522
"ItemKNN (cosine, K=100, history only)",56268.0,0.2803,0.2030,0.3977,0.2406,0.5645,0.2826
"Logistic regression (C=10.0, destination + context, no history)",56268.0,0.5166,0.3500,0.7106,0.4128,0.8768,0.4551
"[ref nb06, uncollapsed data] plain SMLP4Rec",57519.0,0.3123,0.2358,0.4293,0.2733,0.5942,0.3148
"[ref nb06, uncollapsed data] hybrid (SMLP4Rec + prior + sameDest)",57519.0,0.5912,0.4369,0.7631,0.4925,0.9073,0.5293



VALID | unseen, L=0 (first booking)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,50769.0,0.1250,0.0799,0.2100,0.1072,0.3651,0.1460
"ItemKNN (cosine, K=100, history only)",50769.0,0.1250,0.0799,0.2100,0.1072,0.3651,0.1460
"Logistic regression (C=10.0, destination + context, no history)",50769.0,0.5225,0.3555,0.7100,0.4162,0.8758,0.4584
"[ref nb06, uncollapsed data] plain SMLP4Rec",50803.0,0.1251,0.0800,0.2101,0.1073,0.3651,0.1461
"[ref nb06, uncollapsed data] hybrid (SMLP4Rec + prior + sameDest)",50803.0,0.5289,0.3614,0.7158,0.4219,0.8848,0.4648


## 7. Kiểm chứng

Mỗi kiểm tra làm notebook thất bại (assert) nếu không thỏa.

In [9]:
checks = {}
rs = np.random.default_rng(SEED)

# a. no forbidden column reaches the logistic regression
bad = [c for c in LR_FEATURES if any(f in c for f in FORBIDDEN)]
checks["lr_forbidden_features"] = bad
assert not bad

# b. no eval target (warm or cold) is in the training data of either model; training ends before valid starts
eval_src = set(pd.concat([d[k] for d in EVAL.values() for k in d])["src_row"])
checks["eval_rows_in_training_data"] = int(train_rows["src_row"].isin(eval_src).sum())
assert checks["eval_rows_in_training_data"] == 0
checks["training_max_ts_minus_first_valid_target_ts"] = int(train_rows["ts"].max() - rng_t.loc["valid", "min"])
assert checks["training_max_ts_minus_first_valid_target_ts"] < 0

# c. ItemKNN: history @ similarity equals implicit's own recommend() on a sample of test rows
h_te = HIST["test"][0]
idx = np.sort(rs.choice(h_te.shape[0], 2000, replace=False))
dense = (h_te[idx] @ knn_models[K_PICK].similarity).toarray()
ids, sc = knn_models[K_PICK].recommend(np.arange(len(idx)), h_te[idx].tocsr(), N=N_CLUSTERS, filter_already_liked_items=False)
lib = np.zeros_like(dense)
np.put_along_axis(lib, ids, sc.astype(dense.dtype), axis=1)
checks["itemknn_max_abs_diff_vs_recommend"] = float(np.abs(lib - dense).max())
assert checks["itemknn_max_abs_diff_vs_recommend"] < 1e-4


# d. Recall@5 / NDCG@5 recomputed by a plain Python loop (no nbp code) from raw scores on the same sample
def loop_metrics(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    rec = nd = 0.0
    for srow, t in zip(score, y):
        top = sorted(range(N_CLUSTERS), key=lambda c: (-srow[c], -pop[c], c))[:5]
        if t in top:
            rec += 1.0
            nd += 1.0 / np.log2(top.index(t) + 2)
    return rec / len(y), nd / len(y)


y_s = Y["test"]["warm"][idx]
for name, score in ((KNN, dense), (LR, lr_models[C_PICK].predict_proba(X_ev["test"]["warm"][idx]))):
    lp_r, lp_n = loop_metrics(score, y_s)
    r = ranks["test"]["warm"][name][idx]
    fast = (float((r <= 5).mean()), float(((r <= 5) / np.log2(r + 1)).mean()))
    checks[f"loop_vs_fast [{name}]"] = {"loop": [round(lp_r, 6), round(lp_n, 6)], "fast": [round(fast[0], 6), round(fast[1], 6)]}
    assert abs(lp_r - fast[0]) < 1e-9 and abs(lp_n - fast[1]) < 1e-9

# e. controls on test warm rows: shuffled query features (LR), shuffled histories (ItemKNN)
perm = rs.permutation(len(Y["test"]["warm"]))
r_shuf_lr = rank_pop_ties(lr_models[C_PICK].predict_proba(X_ev["test"]["warm"][perm]), Y["test"]["warm"])
r_shuf_knn = rank_pop_ties((h_te[perm] @ knn_models[K_PICK].similarity).toarray(), Y["test"]["warm"])
checks["controls_test_warm"] = {
    "LR real": summarize(ranks["test"]["warm"][LR], ks=(5,)),
    "LR shuffled query features": summarize(r_shuf_lr, ks=(5,)),
    "ItemKNN real": summarize(ranks["test"]["warm"][KNN], ks=(5,)),
    "ItemKNN shuffled histories": summarize(r_shuf_knn, ks=(5,)),
    "global popularity": summarize(ranks["test"]["warm"][GLOB], ks=(5,)),
}
cw = checks["controls_test_warm"]
assert cw["LR shuffled query features"]["ndcg@5"] < cw["LR real"]["ndcg@5"]
assert cw["ItemKNN shuffled histories"]["ndcg@5"] < cw["ItemKNN real"]["ndcg@5"]


# f. ties: optimistic vs pessimistic rank of the target (how far the tie rule can move NDCG@5)
def tie_spread(score: np.ndarray, y: np.ndarray) -> dict:
    st = score[np.arange(len(y)), y][:, None]
    opt = 1 + (score > st).sum(1)
    pes = opt + (score == st).sum(1) - 1
    return {"rows_with_tie_at_target": round(float((pes > opt).mean()), 4),
            "ndcg@5_optimistic": round(float(((opt <= 5) / np.log2(opt + 1)).mean()), 4),
            "ndcg@5_pessimistic": round(float(((pes <= 5) / np.log2(pes + 1)).mean()), 4)}


checks["ties_test_warm"] = {
    KNN: tie_spread((h_te @ knn_models[K_PICK].similarity).toarray(), Y["test"]["warm"]),
    LR: tie_spread(lr_models[C_PICK].predict_proba(X_ev["test"]["warm"]), Y["test"]["warm"]),
}
print(json.dumps(checks, indent=1))
print("all verification checks passed")

{
 "lr_forbidden_features": [],
 "eval_rows_in_training_data": 0,
 "training_max_ts_minus_first_valid_target_ts": -154,
 "itemknn_max_abs_diff_vs_recommend": 0.0,
 "loop_vs_fast [ItemKNN (cosine, K=100, history only)]": {
  "loop": [
   0.2945,
   0.202303
  ],
  "fast": [
   0.2945,
   0.202303
  ]
 },
 "loop_vs_fast [Logistic regression (C=10.0, destination + context, no history)]": {
  "loop": [
   0.5235,
   0.35997
  ],
  "fast": [
   0.5235,
   0.35997
  ]
 },
 "controls_test_warm": {
  "LR real": {
   "n": 213869,
   "recall@5": 0.5222,
   "ndcg@5": 0.3568
  },
  "LR shuffled query features": {
   "n": 213869,
   "recall@5": 0.0784,
   "ndcg@5": 0.0467
  },
  "ItemKNN real": {
   "n": 213869,
   "recall@5": 0.3038,
   "ndcg@5": 0.2091
  },
  "ItemKNN shuffled histories": {
   "n": 213869,
   "recall@5": 0.0838,
   "ndcg@5": 0.0494
  },
  "global popularity": {
   "n": 213869,
   "recall@5": 0.1487,
   "ndcg@5": 0.0944
  }
 },
 "ties_test_warm": {
  "ItemKNN (cosine, K=100, histo

## 8. Lưu

In [10]:
out = {
    "run": "two basic baselines (ItemKNN, multinomial logistic regression) on the plain SMLP4Rec 8/1/1 temporal split",
    "dataset": "expedia (bookings, burst repeats collapsed; item = hotel_cluster)",
    "data_version": "collapsed (expedia_bookings.parquet)",
    "generated_by": "notebooks/hospitality/smlp4rec/07_basic_baselines.ipynb",
    "libraries": {"implicit": implicit.__version__, "scikit-learn": sklearn.__version__},
    "split": {
        "rule": "global temporal 80/10/10 over next-booking targets (load_bookings in this notebook); cold = first bookings inside the valid / test window",
        "counts": {k: int(v) for k, v in b["split"].value_counts().items()},
        "time_range_unix": rng_t.to_dict("index"),
        "training_cut_unix": cut,
        "slice_sizes_vs_reference": size_check,
    },
    "models": {
        KNN: {"library": "implicit.nearest_neighbours.CosineRecommender", "fit_on": "user x cluster booking counts before the cut",
              "input": f"last {MAX_HIST} bookings before the target", "cold_rows": "global popularity",
              "K_grid_valid": knn_valid, "K_pick": K_PICK},
        LR: {"library": "sklearn.linear_model.LogisticRegression(solver='saga'), multinomial", "fit_on": "bookings before the cut",
             "features": LR_FEATURES, "lead_bins": LEAD_BINS, "stay_clip": STAY_CLIP, "epochs": SAGA_EPOCHS, "tol": SAGA_TOL,
             "C_grid_valid": lr_valid, "C_pick": C_PICK},
    },
    "selection": "valid, mean per-row Recall@5 + NDCG@5; ties toward the smaller K / C",
    "tie_rule": "ties broken toward the cluster more popular before the cut",
    "results": results,
    "reference": {"source": "results/week3_implementation/smlprec_expedia_seen_unseen_users.json", "rows": [PLAIN, HYB]},
    "verification": checks,
}
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(out, indent=2, default=str), encoding="utf-8")
for s in EVAL:
    pd.concat([
        pd.DataFrame({"src_row": EVAL[s][k]["src_row"].to_numpy(), "slice": k,
                      **{name: ranks[s][k][name].astype(np.int16) for name in MODELS}})
        for k in ("warm", "cold")
    ]).to_parquet(RANKS_DIR / f"basic_baselines_ranks_{s}.parquet", index=False)
joblib.dump({"knn_similarity": knn_models[K_PICK].similarity, "lr": lr_models[C_PICK], "lr_encoder": enc,
             "pop": pop, "cut": cut}, RANKS_DIR / "basic_baselines_models.joblib")
print("saved", OUT_JSON.relative_to(ROOT), "+ ranks and models in data/interim")

saved results\week4_rebuild\basic_baselines.json + ranks and models in data/interim
